In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
# Copy all original source files from the verified team branch (part 1/4).
import json,hashlib,ast
root=repo/'solution'
assert (root/'swiglu.py').is_file() and (root/'test_kernels.py').is_file()
items=json.loads("[{\"name\":\"__init__.py\",\"sha\":\"73a5f8fc063f671a41dc6a1dc40f567057de2fbc\",\"content\":\"\\\"\\\"\\\"Your solution.\\n\\nThe harness imports this package and calls `apply()` once, in the same process as `soup train`,\\nbefore Soup builds the model and the trainer. Patch whatever you need (transformers, peft,\\nbitsandbytes, Soup itself) so the training run uses your Triton kernels.\\n\\nThe rules in short (the README and the task statement have the full ones):\\n- your own `@triton.jit` kernels, defined in this folder, run in every measured run and together\\n  take >= 5% of the training time; the example's kernels (however renamed or rewritten) and\\n  kernels generated by torch.compile do not count;\\n- no ready-made kernel libraries (unsloth, liger-kernel, xformers, flash-attn, torchao, ...);\\n- the task stays the same: model, data, batches and their order, steps, LoRA, optimizer, Soup's\\n  settings and outputs; no other GPU, no background processes, no tricks with the measurement.\\n\\\"\\\"\\\"\\n\\ndef apply():\\n    from .swiglu import patch_qwen2_mlp\\n    from .rmsnorm import patch_qwen2_rmsnorm\\n    from .integration import patch_training_head\\n    from .settings import (\\n        FUSED_NF4, FROZEN_STORAGE, COMPACT_MLP_ACTIVATIONS,\\n        FUSED_ROTARY, FUSED_LORA_EPILOGUE, FUSED_LORA_DROPOUT,\\n        FUSED_RMSNORM, SHARED_ATTENTION_MASK,\\n    )\\n\\n    if FUSED_NF4:\\n        from .nf4 import patch_bitsandbytes_nf4\\n        patch_bitsandbytes_nf4()\\n    if FROZEN_STORAGE:\\n        from .frozen_storage import patch_frozen_storage\\n        patch_frozen_storage()\\n    if FUSED_LORA_DROPOUT:\\n        from .lora_dropout import patch_lora_dropout\\n        patch_lora_dropout()\\n    if FUSED_LORA_EPILOGUE:\\n        from .lora_epilogue import patch_lora_epilogue\\n        patch_lora_epilogue()\\n    patch_qwen2_mlp()  # Explicitly credited starter example; not original work.\\n    if COMPACT_MLP_ACTIVATIONS:\\n        from .activation_storage import patch_compact_mlp_activations\\n        patch_compact_mlp_activations()\\n    if FUSED_RMSNORM:\\n        patch_qwen2_rmsnorm()\\n    if FUSED_ROTARY:\\n        from .rotary import patch_qwen2_rotary\\n        patch_qwen2_rotary()\\n    if SHARED_ATTENTION_MASK:\\n        from .shared_attention_mask import patch_shared_attention_mask\\n        patch_shared_attention_mask()\\n    # The order is intentional:\\n    # 1. Soup/TRL compute_loss installs the original metrics/rewind contract.\\n    # 2. The independent vocabulary-slice head supplies actual CE/entropy/dH.\\n    # 3. The packed decoder wraps the labelled causal LM forward, keeping\\n    #    native padded-sized PEFT dropout draws and causal attention segments.\\n    # 4. Frozen Q/K/V base GEMM fuses without replacing trainable LoRA.\\n    patch_training_head()\\n    from .hero3_vocab import install_vocab_head\\n    from .hero3_pack import patch_full_decoder_packing\\n    from .hero3_qkv import patch_grouped_qkv\\n    install_vocab_head()\\n    patch_full_decoder_packing()\\n    patch_grouped_qkv()\\n\\n\\n# Independent input-only frozen NF4 MLP. Installed after the packed decoder.\\n_previous_beast_apply = apply\\n\\ndef apply():\\n    _previous_beast_apply()\\n    from .inputonly_mlp import patch_inputonly_mlp\\n    patch_inputonly_mlp()\\n\\n# Combined exact input-only NF4 MLP and real-token CE entropy head.\\n_previous_inputonly_apply=apply\\ndef apply():\\n    _previous_inputonly_apply()\\n    from .valid_head_lite import patch\\n    patch()\\n\\n# Preserve the exact original tied FP16 storage during PEFT preparation.\\n_previous_valid_apply=apply\\ndef apply():\\n    _previous_valid_apply()\\n    from .early_storage import patch_early_storage\\n    patch_early_storage()\\n\\n# Original grouped frozen-NF4 backward; installed after input-only replay.\\n_previous_grouped_apply = apply\\ndef apply():\\n    _previous_grouped_apply()\\n    from .grouped_dx import patch_grouped_dx_runtime_n\\n    patch_grouped_dx_runtime_n()\\n\"},{\"name\":\"activation_offload.py\",\"sha\":\"96c1b61bf1002582e9f4a89a58b93fd460eeec3d\",\"content\":\"\\\"\\\"\\\"Original pipelined CUDA activation spill; T4 admission remains unverified.\\\"\\\"\\\"\\nfrom functools import wraps\\nfrom contextvars import ContextVar\\nfrom threading import local\\nimport os\\nimport torch\\n_tls = local()\\n_active = ContextVar('own_spill', default=None)\\n_stats = dict(packed=0, restored=0, bytes=0, fallback=0, peak_staged=0)\\nclass _Spill:\\n    __slots__ = ('host','ready','stage','shape','dtype','device')\\n    def __init__(self, host, event, stage, t):\\n        self.host,self.ready,self.stage = host,event,stage\\n        self.shape,self.dtype,self.device = tuple(t.shape),t.dtype,t.device\\ndef counters():\\n    return dict(_stats)\\ndef _candidate(t):\\n    if not isinstance(t,torch.Tensor) or not t.is_cuda or t.dtype!=torch.float16 or not t.is_contiguous() or t.ndim not in (2,3) or not t.numel():\\n        return False\\n    rows,width=t.numel()//t.shape[-1],t.shape[-1]\\n    return 512<=rows<=2048 and (width==8960 or (width==1536 and t.ndim==3))\\ndef _stream(device):\\n    d=getattr(_tls,'streams',None)\\n    if d is None:\\n        _tls.streams=d={}\\n    key=(device.type,device.index)\\n    if key not in d:\\n        d[key]=torch.cuda.Stream(device=device)\\n    return d[key]\\ndef _release(session,limit):\\n    compute=torch.cuda.current_stream(session['device'])\\n    pending=session['pending']\\n    while pending and pending[0][0]<=limit:\\n        _,v=pending.pop(0)\\n        compute.wait_event(v.ready)\\n        session['staged']-=v.stage.numel()*v.stage.element_size()\\n        v.stage=None\\ndef _pack(t):\\n    s=_active.get()\\n    if s is None or not _candidate(t):\\n        return t\\n    try:\\n        host=torch.empty(t.shape,dtype=t.dtype,device='cpu',pin_memory=True)\\n    except (RuntimeError,MemoryError):\\n        _stats['fallback']+=1\\n        return t\\n    stage=torch.empty_like(t)\\n    stage.copy_(t)\\n    copy=_stream(t.device)\\n    compute=torch.cuda.current_stream(t.device)\\n    with torch.cuda.stream(copy):\\n        copy.wait_stream(compute)\\n        host.copy_(stage,non_blocking=True)\\n        event=torch.cuda.Event(blocking=False)\\n        event.record(copy)\\n    v=_Spill(host,event,stage,t)\\n    s['pending'].append((s['depth'],v))\\n    size=t.numel()*t.element_size()\\n    s['staged']+=size\\n    _stats['packed']+=1\\n    _stats['bytes']+=size\\n    _stats['peak_staged']=max(_stats['peak_staged'],s['staged'])\\n    if s['staged']>192*1024**2:\\n        _release(s,s['depth']-1)\\n    return v\\ndef _unpack(v):\\n    if not isinstance(v,_Spill):\\n        return v\\n    compute=torch.cuda.current_stream(v.device)\\n    compute.wait_event(v.ready)\\n    x=torch.empty(v.shape,device=v.device,dtype=v.dtype)\\n    x.copy_(v.host,non_blocking=True)\\n    _stats['restored']+=1\\n    return x\\ndef patch_decoder_activation_offload():\\n    from transformers.models.qwen2.modeling_qwen2 import Qwen2Model,Qwen2DecoderLayer\\n    previous=Qwen2Model.forward\\n    if getattr(previous,'_soup_activation_spill',False):\\n        return True\\n    previous_layer=Qwen2DecoderLayer.forward\\n    @wraps(previous_layer)\\n    def layer_forward(self,*args,**kwargs):\\n        s=_active.get()\\n        if s is None:\\n            return previous_layer(self,*args,**kwargs)\\n        s['depth']+=1\\n        try:\\n            return previous_layer(self,*args,**kwargs)\\n        finally:\\n            _release(s,s['depth']-2)\\n    @wraps(previous)\\n    def forward(self,*args,**kwargs):\\n        if not self.training or not torch.is_grad_enabled() or getattr(self,'gradient_checkpointing',False) or not torch.cuda.is_available() or os.getenv('SOUP_DISABLE_ACTIVATION_SPILL')=='1':\\n            return previous(self,*args,**kwargs)\\n        ids=kwargs.get('input_ids',args[0] if args else None)\\n        if not isinstance(ids,torch.Tensor) or not ids.is_cuda or ids.ndim!=2 or ids.numel()<1024 or _active.get() is not None:\\n            return previous(self,*args,**kwargs)\\n        s={'device':ids.device,'pending':[],'depth':0,'staged':0}\\n        token=_active.set(s)\\n        try:\\n            with torch.autograd.graph.saved_tensors_hooks(_pack,_unpack):\\n                return previous(self,*args,**kwargs)\\n        finally:\\n            _release(s,s['depth'])\\n            _active.reset(token)\\n    forward._soup_activation_spill=True\\n    Qwen2DecoderLayer.forward=layer_forward\\n    Qwen2Model.forward=forward\\n    return True\\n\"},{\"name\":\"activation_storage.py\",\"sha\":\"355f849f2f33be76e36e9cb3d5f18e301047a5b6\",\"content\":\"\\\"\\\"\\\"Optional lossless storage of frozen NF4 MLP intermediates for backward.\\n\\nThe pinned bitsandbytes Linear4bit consumes its configured FP16 compute input,\\nreturns FP16 GEMM output, and casts that output back to the incoming FP32 dtype.\\nFor ordinary frozen Qwen2 gate/up projections this proves their FP32 outputs\\nare exactly FP16-representable. Passing two separately cast FP16 inputs keeps\\nthose outputs in FP16 and avoids the round trip through global FP32 storage.\\nThe starter kernels still perform activation arithmetic in FP32 registers;\\nthe activation output is FP32, and each projection gradient is rounded to\\nFP16 at the same point as in the original Linear4bit casting graph.\\n\\nBoth Triton functions used here are the UNCHANGED, CREDITED STARTER SWIGLU\\nKERNELS. This storage optimization introduces NO original/countable kernel.\\nIn particular, a new mixed-pointer backward specialization is still the\\nstarter example and must not count toward the 5% original-kernel threshold.\\n\\\"\\\"\\\"\\n\\nfrom functools import wraps\\nfrom importlib import metadata\\n\\nimport torch\\nimport triton\\n\\nfrom .swiglu import (\\n    BLOCK,\\n    example_swiglu_fwd_kernel,\\n    example_swiglu_bwd_kernel,\\n    swiglu,\\n)\\n\\n\\nclass _CompactSwiGLU(torch.autograd.Function):\\n    @staticmethod\\n    def forward(ctx, gate, up):\\n        gate, up = gate.contiguous(), up.contiguous()\\n        ctx.input_dtype = gate.dtype\\n        # Even when the projection outputs remain FP16, the original FP32\\n        # SwiGLU result must not be rounded before the down projection's cast.\\n        output = torch.empty_like(gate, dtype=torch.float32)\\n        n = gate.numel()\\n        example_swiglu_fwd_kernel[(triton.cdiv(n, BLOCK),)](\\n            gate, up, output, n, BLOCK=BLOCK,\\n        )\\n        # FP16 inputs are saved by reference. For the compatibility FP32 path,\\n        # these casts are lossless only under the checked projection dataflow.\\n        # No original FP32 gate/up tensor is retained by this autograd context.\\n        ctx.save_for_backward(gate.to(torch.float16), up.to(torch.float16))\\n        return output\\n\\n    @staticmethod\\n    def backward(ctx, grad):\\n        gate_half, up_half = ctx.saved_tensors\\n        # Preserve the original FP32 upstream gradient. Casting to the saved\\n        # tensor dtype here would round before differentiation and be wrong.\\n        grad = grad.to(torch.float32).contiguous()\\n        dgate = torch.empty_like(gate_half, dtype=ctx.input_dtype)\\n        dup = torch.empty_like(up_half, dtype=ctx.input_dtype)\\n        n = gate_half.numel()\\n        # The credited kernel evaluates each derivative in FP32 registers.\\n        # With FP16 projection outputs, storing that derivative as FP16 fuses\\n        # the original output-upcast's backward conversion. For FP32 inputs,\\n        # retain FP32 derivative outputs as the earlier storage wrapper did.\\n        # Neither mode reconstructs global FP32 gate/up activations.\\n        example_swiglu_bwd_kernel[(triton.cdiv(n, BLOCK),)](\\n            grad, gate_half, up_half, dgate, dup, n, BLOCK=BLOCK,\\n        )\\n        return dgate, dup\\n\\n\\ndef compact_swiglu(gate, up):\\n    \\\"\\\"\\\"Internal primitive: matching FP16 inputs or exact FP16 values in FP32.\\n\\n    Always returns an FP32 activation. The production patch uses FP16 input\\n    tensors directly. The compatibility FP32 path does not scan values;\\n    arbitrary FP32 inputs violate its exact-representability precondition.\\n    \\\"\\\"\\\"\\n    if (gate.dtype not in (torch.float16, torch.float32) or up.dtype != gate.dtype\\n            or gate.shape != up.shape or gate.device != up.device\\n            or gate.numel() == 0):\\n        raise ValueError(\\\"requires matching nonempty FP16 values or their exact FP32 representations\\\")\\n    return _CompactSwiGLU.apply(gate, up)\\n\\n\\ndef _is_cuda_tensor(tensor):\\n    return tensor.is_cuda\\n\\n\\ndef _eligible_mlp(module, x, linear_class, linear_forward):\\n    if (not isinstance(x, torch.Tensor) or not _is_cuda_tensor(x)\\n            or x.dtype != torch.float32 or x.ndim < 2 or not x.is_contiguous()\\n            or not x.numel() or not x.requires_grad or not module.training or not torch.is_grad_enabled()\\n            or getattr(getattr(module, \\\"config\\\", None), \\\"hidden_act\\\", None) != \\\"silu\\\"):\\n        return False\\n    # Refuse output-changing hooks or overrides: the representability proof\\n    # depends on the documented, unmodified Linear4bit forward formula.\\n    hooks = torch.nn.modules.module\\n    if (hooks._global_forward_hooks or hooks._global_forward_pre_hooks\\n            or hooks._global_backward_hooks or hooks._global_backward_pre_hooks):\\n        return False\\n    projections = [getattr(module, name, None) for name in (\\\"gate_proj\\\", \\\"up_proj\\\", \\\"down_proj\\\")]\\n    for layer in projections:\\n        if (type(layer) is not linear_class or type(layer).forward is not linear_forward\\n                or \\\"forward\\\" in layer.__dict__ or hasattr(layer, \\\"_hf_hook\\\")\\n                or layer._forward_hooks or layer._forward_pre_hooks\\n                or layer._backward_hooks or layer._backward_pre_hooks\\n                or layer.compute_dtype != torch.float16 or not layer.compute_type_is_set\\n                or layer.bias is not None or layer.weight.requires_grad\\n                or layer.weight.dtype != torch.uint8\\n                or not _is_cuda_tensor(layer.weight) or layer.weight.device != x.device):\\n            return False\\n        state = getattr(layer.weight, \\\"quant_state\\\", None)\\n        if (state is None or state.quant_type != \\\"nf4\\\" or state.dtype != torch.float16\\n                or tuple(state.shape) != (layer.out_features, layer.in_features)):\\n            return False\\n    gate, up, down = projections\\n    return (gate.in_features == up.in_features == x.shape[-1]\\n            and gate.out_features == up.out_features == down.in_features\\n            and down.out_features == x.shape[-1])\\n\\n\\ndef patch_compact_mlp_activations():\\n    \\\"\\\"\\\"Install after the starter MLP patch; unsupported calls keep that path.\\\"\\\"\\\"\\n    if metadata.version(\\\"bitsandbytes\\\") != \\\"0.50.2\\\" or metadata.version(\\\"transformers\\\") != \\\"5.16.1\\\":\\n        return False\\n    import bitsandbytes as bnb\\n    from transformers.models.qwen2.modeling_qwen2 import Qwen2MLP\\n\\n    original = Qwen2MLP.forward\\n    if getattr(original, \\\"_soup_candidate_compact_activations\\\", False):\\n        return True\\n    linear_class = bnb.nn.Linear4bit\\n    linear_forward = linear_class.forward\\n\\n    @wraps(original)\\n    def forward(self, x):\\n        if not _eligible_mlp(self, x, linear_class, linear_forward):\\n            return original(self, x)\\n        # Keep these casts SEPARATE. Sharing one FP16 input would sum the two\\n        # input-gradient branches in FP16 before upcasting to FP32, unlike the\\n        # original graph, which upcasts each branch first and then adds them.\\n        gate = self.gate_proj(x.to(torch.float16))\\n        up = self.up_proj(x.to(torch.float16))\\n        if (gate.dtype == up.dtype == torch.float16 and gate.shape == up.shape\\n                and gate.device == up.device and gate.numel()):\\n            activation = compact_swiglu(gate, up)\\n        else:\\n            # Unmodified guarded projections produce FP16. If that contract\\n            # is violated, do not rerun projections; preserve FP32 arithmetic.\\n            activation = swiglu(gate.float(), up.float())\\n        return self.down_proj(activation)\\n\\n    forward._soup_candidate_compact_activations = True\\n    Qwen2MLP.forward = forward\\n    return True\\n\"},{\"name\":\"cached_head.py\",\"sha\":\"ca02a0f9c5ef458c80f75d857de45ae50fbdc7a8\",\"content\":\"\\\"\\\"\\\"Private native FP16 logits, fused summaries, bounded native CE backward.\\n\\nThis composes the user's Round 7 native-head/control-gradient result with the\\noriginal tiled statistics from their build candidate.  It does NOT replay the\\nhead projection, change its GEMM geometry, or reconstruct its scaled gradient\\nwith an approximate exponential.  The ordinary head autograd graph remains\\nresponsible for the single full-shape hidden-gradient GEMM.\\n\\nOnly compact forward reductions are original Triton work here.  Native CE\\nbackward is intentionally retained as a numerical control.  No GPU runtime or\\npeak-memory improvement is implied by the structural allocation reduction.\\n\\\"\\\"\\\"\\n\\nimport torch\\nimport torch.nn.functional as F\\nimport triton\\nimport triton.language as tl\\n\\nfrom .streamed_head import (\\n    HeadResult,\\n    NATIVE_NORMALIZER_THRESHOLD,\\n    VOCAB_TILE,\\n    _native_row_statistics,\\n    soup_head_partial_statistics,\\n)\\n\\n\\n@triton.jit\\ndef soup_cached_head_finish(\\n    scores, targets, partial_max, partial_sum, partial_centered_sum,\\n    partial_argmax, token_loss, token_entropy, predictions, exceptional,\\n    VOCAB: tl.constexpr, TILES: tl.constexpr, MERGE_BLOCK: tl.constexpr,\\n    SATURATION_THRESHOLD: tl.constexpr,\\n):\\n    \\\"\\\"\\\"Merge our tile summaries; store only consumer outputs and a guard.\\\"\\\"\\\"\\n    row = tl.program_id(0)\\n    tile = tl.arange(0, MERGE_BLOCK)\\n    active = tile < TILES\\n    offset = row * TILES + tile\\n    local_max = tl.load(partial_max + offset, active, other=-float(\\\"inf\\\"))\\n    maximum = tl.max(local_max, 0)\\n    delta = tl.where(active, local_max - maximum, 0.0)\\n    factor = tl.exp(delta)\\n    local_sum = tl.load(partial_sum + offset, active, other=0.0)\\n    local_centered = tl.load(partial_centered_sum + offset, active, other=0.0)\\n    total = tl.sum(factor * local_sum, 0)\\n    centered = tl.sum(factor * (local_centered + delta * local_sum), 0)\\n    log_total = tl.log(total)\\n    entropy = log_total - centered / total\\n    local_argmax = tl.load(partial_argmax + offset, active, other=2147483647)\\n    prediction = tl.min(tl.where(active & (local_max == maximum),\\n                                 local_argmax, 2147483647), 0)\\n    target = tl.load(targets + row)\\n    supervised = target != -100\\n    in_bounds = (target >= 0) & (target < VOCAB)\\n    # Invalid labels are rejected by the asynchronous host wrapper assertion;\\n    # this bound also makes the pointer safe independently of its scheduling.\\n    safe_target = tl.minimum(tl.maximum(target, 0), VOCAB - 1)\\n    target_score = tl.load(scores + row * VOCAB + safe_target,\\n                           supervised & in_bounds, other=0.0).to(tl.float32)\\n    ce = tl.where(supervised, -((target_score - maximum) - log_total), 0.0)\\n    ce = tl.where(supervised & ~in_bounds, float(\\\"nan\\\"), ce)\\n    finite_max = (maximum == maximum) & (tl.abs(maximum) != float(\\\"inf\\\"))\\n    finite_log = (log_total == log_total) & (tl.abs(log_total) != float(\\\"inf\\\"))\\n    finite_entropy = (entropy == entropy) & (tl.abs(entropy) != float(\\\"inf\\\"))\\n    guarded = ((log_total <= SATURATION_THRESHOLD) | ~finite_max\\n               | ~finite_log | ~finite_entropy)\\n    tl.store(token_loss + row, ce)\\n    tl.store(token_entropy + row, entropy)\\n    tl.store(predictions + row, prediction)\\n    tl.store(exceptional + row, guarded)\\n\\n\\ndef _summaries(flat, targets, chunk_size):\\n    rows, vocab = flat.shape\\n    tiles = triton.cdiv(vocab, VOCAB_TILE)\\n    maximum = torch.empty((rows, tiles), device=flat.device, dtype=torch.float32)\\n    total = torch.empty_like(maximum)\\n    centered = torch.empty_like(maximum)\\n    winner = torch.empty((rows, tiles), device=flat.device, dtype=torch.int32)\\n    loss = torch.empty(rows, device=flat.device, dtype=torch.float32)\\n    entropy = torch.empty_like(loss)\\n    predictions = torch.empty(rows, device=flat.device, dtype=torch.long)\\n    exceptional = torch.empty(rows, device=flat.device, dtype=torch.bool)\\n    soup_head_partial_statistics[(rows, tiles)](\\n        flat, maximum, total, centered, winner,\\n        VOCAB=vocab, TILES=tiles, BLOCK=VOCAB_TILE, num_warps=4,\\n    )\\n    soup_cached_head_finish[(rows,)](\\n        flat, targets, maximum, total, centered, winner,\\n        loss, entropy, predictions, exceptional,\\n        VOCAB=vocab, TILES=tiles, MERGE_BLOCK=triton.next_power_of_2(tiles),\\n        SATURATION_THRESHOLD=NATIVE_NORMALIZER_THRESHOLD, num_warps=4,\\n    )\\n    # Native entropy deliberately preserves NaN from e.g. 0*log(0), including\\n    # masked/ignored rows.  Near-saturation falls back for forward accuracy.\\n    # This size-discovery synchronization is a real cost to measure on T4.\\n    exceptional_rows = exceptional.nonzero(as_tuple=False).flatten()\\n    for start in range(0, exceptional_rows.numel(), chunk_size):\\n        selected = exceptional_rows[start:start + chunk_size]\\n        native = _native_row_statistics(flat.index_select(0, selected),\\n                                         targets.index_select(0, selected))\\n        for output, values in zip((loss, entropy, predictions), native):\\n            output.index_copy_(0, selected, values)\\n    return loss, entropy, predictions\\n\\n\\nclass _CachedLogitsCrossEntropy(torch.autograd.Function):\\n    @staticmethod\\n    def forward(ctx, logits, labels, num_items_in_batch, chunk_size):\\n        batch, length, vocab = logits.shape\\n        flat = logits.view(-1, vocab)\\n        shifted = F.pad(labels, (0, 1), value=-100)[..., 1:].contiguous()\\n        targets = shifted.view(-1)\\n        # Value checking stays asynchronous on CUDA; no host .item() here.\\n        torch._assert_async(((targets == -100) | ((targets >= 0) & (targets < vocab))).all(),\\n                            \\\"causal target index is out of bounds\\\")\\n        token_loss, entropy, predictions = _summaries(flat, targets, chunk_size)\\n        if num_items_in_batch is None:\\n            divisor = (targets != -100).sum().to(torch.float32)\\n        else:\\n            divisor = torch.as_tensor(num_items_in_batch, device=logits.device,\\n                                      dtype=torch.float32).detach().reshape(())\\n        # Never clamp: native all-ignored mean CE is NaN with zero finite-row\\n        # gradients; explicit zero/negative divisors preserve their semantics.\\n        loss = token_loss.sum() / divisor\\n        ctx.save_for_backward(logits, targets, divisor)\\n        ctx.chunk_size = chunk_size\\n        ctx.set_materialize_grads(False)\\n        token_loss = token_loss.reshape(batch, length)\\n        entropy = entropy.reshape(batch, length)\\n        predictions = predictions.reshape(batch, length)\\n        ctx.mark_non_differentiable(token_loss, entropy, predictions)\\n        return loss, token_loss, entropy, predictions\\n\\n    @staticmethod\\n    def backward(ctx, grad_loss, _token_loss, _entropy, _predictions):\\n        if grad_loss is None:\\n            return None, None, None, None\\n        logits, targets, divisor = ctx.saved_tensors\\n        flat = logits.reshape(-1, logits.shape[-1])\\n        higher_order = torch.is_grad_enabled()\\n        # Ordinary training writes one full FP16 dZ; native FP32 CE workspaces\\n        # exist for just one chunk.  The original head receives this tensor and\\n        # executes its unmodified full-shape backward GEMM once.\\n        dz = None if higher_order else torch.empty_like(flat)\\n        pieces = [] if higher_order else None\\n        with torch.enable_grad(), torch.autocast(device_type=logits.device.type, enabled=False):\\n            for start in range(0, flat.shape[0], ctx.chunk_size):\\n                stop = min(start + ctx.chunk_size, flat.shape[0])\\n                part = flat[start:stop]\\n                leaf = part if higher_order else part.detach().requires_grad_(True)\\n                native_loss = F.cross_entropy(leaf.float(), targets[start:stop],\\n                                               ignore_index=-100, reduction=\\\"sum\\\") / divisor\\n                gradient, = torch.autograd.grad(native_loss, leaf,\\n                                                 grad_outputs=grad_loss.float(),\\n                                                 create_graph=higher_order)\\n                if higher_order:\\n                    pieces.append(gradient)\\n                else:\\n                    dz[start:stop].copy_(gradient)\\n                # In the ordinary path no chunk graph survives to the next\\n                # iteration.  Ignored nonfinite rows are never skipped.\\n        if higher_order:\\n            dz = torch.cat(pieces, dim=0)\\n        return dz.reshape(logits.shape), None, None, None\\n\\n\\ndef cached_logits_ce(logits, labels, num_items_in_batch=None, chunk_size=128):\\n    \\\"\\\"\\\"Return differentiable scalar loss and detached true per-token metrics.\\n\\n    ``logits`` must be the unchanged native head's contiguous FP16 (B,T,V)\\n    output.  No copy of that tensor is retained.  ``labels`` use causal shift\\n    and ignore index -100; the last position contributes entropy but no CE.\\n    Explicit denominators are global, not per chunk.  Actual upstream scaling\\n    is applied by native autograd before the FP32-to-FP16 gradient cast.\\n    \\\"\\\"\\\"\\n    if logits.ndim != 3 or labels.shape != logits.shape[:2] or min(logits.shape) < 1:\\n        raise ValueError(\\\"expected nonempty logits (B,T,V) and labels (B,T)\\\")\\n    if logits.dtype != torch.float16 or labels.dtype != torch.long:\\n        raise TypeError(\\\"cached head requires FP16 logits and int64 labels\\\")\\n    if logits.device != labels.device or not logits.is_contiguous():\\n        raise ValueError(\\\"cached logits must be contiguous and share the labels' device\\\")\\n    if not isinstance(chunk_size, int) or isinstance(chunk_size, bool) or chunk_size < 1:\\n        raise ValueError(\\\"chunk_size must be a positive integer\\\")\\n    if isinstance(num_items_in_batch, torch.Tensor) and num_items_in_batch.numel() != 1:\\n        raise ValueError(\\\"num_items_in_batch must be scalar\\\")\\n    return HeadResult(*_CachedLogitsCrossEntropy.apply(logits, labels, num_items_in_batch, chunk_size))\\n\"},{\"name\":\"early_storage.py\",\"sha\":\"5d3e041cf0d9ec6fbb5f3e98a44fadd5d8cdb4c2\",\"content\":\"\\\"\\\"\\\"Avoid PEFT's transient full-size FP32 tied embedding in Qwen2 NF4 preparation.\\n\\nThe model starts with an exact FP16 tied frozen base embedding. The original\\nPEFT preparation freezes parameters and upcasts nonquantized tensors. For this\\none guarded parameter, run PEFT with an empty temporary tensor, then restore\\nthe original FP16 bytes, the same Parameter identity and tying, and the\\nexisting exact FP32 lookup bridge. All other PEFT operations run normally.\\n\\\"\\\"\\\"\\nfrom functools import wraps\\nimport inspect\\nimport torch\\nfrom . import frozen_storage\\n\\n_STATS = {'installed': 0, 'fallback': 0}\\n\\ndef counters():\\n    return dict(_STATS)\\n\\ndef patch_early_storage():\\n    import peft\\n    parent = peft.prepare_model_for_kbit_training\\n    if getattr(parent, '_soup_early_tied_half', False):\\n        return True\\n    signature = inspect.signature(parent)\\n\\n    @wraps(parent)\\n    def prepare(model, *args, **kwargs):\\n        bound = signature.bind(model, *args, **kwargs)\\n        bound.apply_defaults()\\n        candidate = frozen_storage._capture_candidate(\\n            model, bound.arguments.get('use_gradient_checkpointing', True))\\n        if candidate is None:\\n            _STATS['fallback'] += 1\\n            return parent(model, *args, **kwargs)\\n        param = candidate.parameter\\n        original_half = candidate.half_storage\\n        # Original PEFT sees the exact same Parameter and module tree but an\\n        # empty FP16 data allocation for the one frozen tied weight only.\\n        # Its parameter-freezing and optional hooks still run unchanged.\\n        param.data = original_half.new_empty((0,))\\n        try:\\n            prepared = parent(model, *args, **kwargs)\\n        finally:\\n            # Restore even if PEFT raises; no invalid tied weights escape.\\n            param.data = original_half\\n        if (prepared is not candidate.model\\n            or candidate.embedding.weight is not param\\n            or candidate.head.weight is not param\\n            or param.requires_grad\\n            or tuple(param.shape) != tuple(original_half.shape)\\n            or param.dtype != torch.float16):\\n            raise RuntimeError('Early tied-embedding preparation violated model identity')\\n        frozen_storage._install_storage(candidate)\\n        _STATS['installed'] += 1\\n        return prepared\\n\\n    prepare._soup_early_tied_half = True\\n    peft.prepare_model_for_kbit_training = prepare\\n    return True\\n\"},{\"name\":\"frozen_storage.py\",\"sha\":\"8758837e1f3fc775abf677b629cf79f6061dd7e6\",\"content\":\"\\\"\\\"\\\"Optional lossless storage for a previously FP16 tied embedding/output head.\\n\\nSoup loads the frozen base in FP16 on T4; PEFT k-bit preparation then converts\\nnon-quantized parameters to FP32. This experiment records the actual FP16\\nstorage *before* that conversion, runs unmodified preparation, and checks that\\nthe resulting frozen tied parameter contains exactly the same values before\\nrestoring the smaller storage. It never compresses arbitrary FP32 values.\\n\\nEmbedding lookup still returns FP32, preserving the decoder's input dtype.\\nOrdinary output-head calls use FP32 operands outside autocast; FP16 autocast\\nand the streamed training head can consume the retained half storage directly.\\nFull base state_dict entries intentionally use the compact storage dtype;\\ntheir names, shapes, values and parameter tying are preserved. Adapter tensors\\nare not touched. This option is disabled by default pending physical T4 tests.\\n\\\"\\\"\\\"\\n\\nfrom dataclasses import dataclass\\nfrom functools import wraps\\nfrom importlib import metadata\\nimport inspect\\nfrom types import MethodType\\n\\nimport torch\\nimport torch.nn.functional as F\\nimport triton\\nimport triton.language as tl\\n\\n\\n@triton.jit(do_not_specialize=[\\\"n_elements\\\"])\\ndef soup_frozen_embedding_forward(\\n    TOKEN_IDS, WEIGHT, OUTPUT, n_elements,\\n    WIDTH: tl.constexpr, BLOCK: tl.constexpr,\\n):\\n    \\\"\\\"\\\"Gather exact FP16-representable frozen weights into FP32 activations.\\\"\\\"\\\"\\n    element = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)\\n    active = element < n_elements\\n    token = tl.load(TOKEN_IDS + element // WIDTH, active, other=0)\\n    value = tl.load(WEIGHT + token * WIDTH + element % WIDTH, active, other=0)\\n    tl.store(OUTPUT + element, value.to(tl.float32), active)\\n\\n\\ndef frozen_embedding(weight, token_ids):\\n    \\\"\\\"\\\"Direct kernel entry point; CPU is supported by Triton's interpreter.\\\"\\\"\\\"\\n    if (weight.ndim != 2 or weight.dtype != torch.float16\\n            or not weight.is_contiguous() or weight.requires_grad):\\n        raise ValueError(\\\"expected a contiguous frozen FP16 embedding matrix\\\")\\n    if token_ids.dtype not in (torch.int32, torch.int64):\\n        raise TypeError(\\\"embedding indices must be int32 or int64\\\")\\n    if token_ids.device != weight.device:\\n        raise ValueError(\\\"embedding matrix and indices must share a device\\\")\\n    token_ids = token_ids.contiguous()\\n    output = torch.empty((*token_ids.shape, weight.shape[1]), dtype=torch.float32,\\n                         device=weight.device)\\n    if output.numel():\\n        soup_frozen_embedding_forward[(triton.cdiv(output.numel(), 512),)](\\n            token_ids, weight, output, output.numel(), WIDTH=weight.shape[1],\\n            BLOCK=512, num_warps=4,\\n        )\\n    return output\\n\\n\\n@dataclass(frozen=True)\\nclass _Candidate:\\n    model: object\\n    embedding: object\\n    head: object\\n    parameter: object\\n    half_storage: object\\n\\n\\ndef _supported_device(weight):\\n    return weight.is_cuda\\n\\n\\ndef _capture_candidate(model, use_gradient_checkpointing):\\n    from transformers.models.qwen2.modeling_qwen2 import Qwen2ForCausalLM\\n\\n    if (type(model) is not Qwen2ForCausalLM\\n            or not getattr(model, \\\"is_loaded_in_4bit\\\", False)\\n            or getattr(model, \\\"is_loaded_in_8bit\\\", False)\\n            or use_gradient_checkpointing\\n            or getattr(model, \\\"_candidate_frozen_storage\\\", False)):\\n        return None\\n    embedding, head = model.get_input_embeddings(), model.get_output_embeddings()\\n    if (type(embedding) is not torch.nn.Embedding or type(head) is not torch.nn.Linear\\n            or head.bias is not None or embedding.max_norm is not None\\n            or embedding.sparse or embedding.scale_grad_by_freq\\n            or \\\"forward\\\" in embedding.__dict__ or \\\"forward\\\" in head.__dict__\\n            or hasattr(embedding, \\\"_hf_hook\\\") or hasattr(head, \\\"_hf_hook\\\")):\\n        return None\\n    weight = embedding.weight\\n    if (weight is not head.weight or weight.dtype != torch.float16\\n            or not _supported_device(weight) or not weight.is_contiguous()\\n            or weight.ndim != 2 or not weight.numel()\\n            or tuple(weight.shape) != (embedding.num_embeddings, embedding.embedding_dim)\\n            or tuple(weight.shape) != (head.out_features, head.in_features)\\n            or not getattr(model.config, \\\"tie_word_embeddings\\\", False)):\\n        return None\\n    # A detached view keeps the original half allocation alive while PEFT\\n    # replaces parameter.data with an upcast allocation. It consumes no RNG.\\n    return _Candidate(model, embedding, head, weight, weight.detach())\\n\\n\\ndef _unchanged_after_preparation(candidate, model):\\n    c = candidate\\n    weight = c.parameter\\n    if (model is not c.model or model.get_input_embeddings() is not c.embedding\\n            or model.get_output_embeddings() is not c.head\\n            or c.embedding.weight is not weight or c.head.weight is not weight\\n            or weight.requires_grad or weight.dtype != torch.float32\\n            or weight.device != c.half_storage.device\\n            or weight.shape != c.half_storage.shape or not weight.is_contiguous()):\\n        return False\\n    # Bounded verification, once at setup: do not allocate another full FP32\\n    # vocabulary matrix. Equal accepts exact FP16 round trips only; NaNs fail.\\n    current, original = weight.detach().view(-1), c.half_storage.view(-1)\\n    for start in range(0, current.numel(), 4 * 1024 * 1024):\\n        end = min(start + 4 * 1024 * 1024, current.numel())\\n        if not torch.equal(current[start:end], original[start:end].float()):\\n            return False\\n    return True\\n\\n\\n@wraps(torch.nn.Embedding.forward)\\ndef _embedding_forward(module, token_ids):\\n    parameter = module.weight\\n    if parameter.dtype != torch.float16:\\n        return torch.nn.Embedding.forward(module, token_ids)\\n    if parameter.requires_grad:\\n        raise RuntimeError(\\\"losslessly packed embedding must remain frozen\\\")\\n    if parameter.is_cuda:\\n        return frozen_embedding(parameter, token_ids)\\n    # Device moves after setup preserve ordinary lookup semantics. The\\n    # production installer itself only captures CUDA models.\\n    return torch.nn.Embedding.forward(module, token_ids).float()\\n\\n\\n@wraps(torch.nn.Linear.forward)\\ndef _head_forward(module, hidden_states):\\n    parameter = module.weight\\n    if parameter.dtype != torch.float16:\\n        return torch.nn.Linear.forward(module, hidden_states)\\n    if parameter.requires_grad:\\n        raise RuntimeError(\\\"losslessly packed output head must remain frozen\\\")\\n    device_type = hidden_states.device.type\\n    if (torch.is_autocast_enabled(device_type)\\n            and torch.get_autocast_dtype(device_type) == torch.float16):\\n        return torch.nn.Linear.forward(module, hidden_states)\\n    # Outside the target AMP path, reproduce the original FP32 weight\\n    # operand, including the usual mixed-dtype error for FP16 inputs.\\n    return F.linear(hidden_states, parameter.float(), module.bias)\\n\\n\\ndef _install_storage(candidate):\\n    c = candidate\\n    # Keep the same Parameter object and its shared ownership. No parameter or\\n    # buffer is added, no task/model config is changed, and normal Module hooks\\n    # (including enable_input_require_grads) still surround each forward.\\n    c.parameter.data = c.half_storage\\n    # Global methods avoid retaining the full candidate or original modules\\n    # in closures, and allow deepcopy to bind each method to the copied module.\\n    c.embedding.forward = MethodType(_embedding_forward, c.embedding)\\n    c.head.forward = MethodType(_head_forward, c.head)\\n    c.model.__dict__[\\\"_candidate_frozen_storage\\\"] = True\\n    c.model.__dict__.pop(\\\"_candidate_head_cast\\\", None)\\n\\n\\ndef patch_frozen_storage():\\n    \\\"\\\"\\\"Install the optional preparation hook for the audited pinned stack.\\\"\\\"\\\"\\n    import peft\\n\\n    if metadata.version(\\\"peft\\\") != \\\"0.20.0\\\" or metadata.version(\\\"transformers\\\") != \\\"5.16.1\\\":\\n        return False\\n    original = peft.prepare_model_for_kbit_training\\n    if getattr(original, \\\"_soup_candidate_frozen_storage\\\", False):\\n        return True\\n    signature = inspect.signature(original)\\n\\n    @wraps(original)\\n    def prepare(model, *args, **kwargs):\\n        bound = signature.bind(model, *args, **kwargs)\\n        bound.apply_defaults()\\n        candidate = _capture_candidate(model, bound.arguments[\\\"use_gradient_checkpointing\\\"])\\n        prepared = original(model, *args, **kwargs)\\n        if candidate is not None and _unchanged_after_preparation(candidate, prepared):\\n            _install_storage(candidate)\\n        return prepared\\n\\n    prepare._soup_candidate_frozen_storage = True\\n    peft.prepare_model_for_kbit_training = prepare\\n    return True\\n\"},{\"name\":\"grouped_dx.py\",\"sha\":\"df39f8c72dbe10114b7c1874ba65ebe9e24f01f2\",\"content\":\"\\\"\\\"\\\"Independent grouped frozen-NF4 MLP input-gradient backward for Soup/T4.\\n\\nAlgorithms informed by the team's SOUP-NANO-002 + HERO4-022/023 research,\\nnot copied from public competitor kernels. Existing input-only MLP *forward*\\nis unchanged. Backward reconstructs frozen gate/up weights into one contiguous\\nbuffer, computes both projections in one cuBLAS GEMM, overwrites the gate/up\\nactivation buffer with both gradients using one original Triton kernel, and\\ncomputes input gradients in one GEMM.\\n\\nThe final grouped FP16 dot has a different reduction order from two separate\\ndots; it must pass full Soup loss/eval checks. This is not bitwise-exact parity.\\n\\\"\\\"\\\"\\nfrom functools import wraps\\n\\nimport torch\\nimport triton\\nimport triton.language as tl\\n\\nfrom .inputonly_mlp import _InputOnly\\n\\nCOUNTS = {\\\"grouped\\\": 0, \\\"fallback\\\": 0, \\\"elements\\\": 0}\\n\\n\\n@triton.jit(do_not_specialize=[\\\"TOTAL\\\"])\\ndef _soup_inplace_grouped_swiglu_vjp(\\n    GU, DH, TOTAL,\\n    INTER: tl.constexpr, BLOCK: tl.constexpr,\\n):\\n    idx = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)\\n    ok = idx < TOTAL\\n    row = idx // INTER\\n    col = idx % INTER\\n    gate_ptr = GU + row * (2 * INTER) + col\\n    up_ptr = gate_ptr + INTER\\n    gate = tl.load(gate_ptr, mask=ok, other=0).to(tl.float32)\\n    up = tl.load(up_ptr, mask=ok, other=0).to(tl.float32)\\n    dy = tl.load(DH + idx, mask=ok, other=0).to(tl.float32)\\n    sig = tl.sigmoid(gate)\\n    act = gate * sig\\n    d_gate = ((dy * up) * sig) * (1.0 + gate * (1.0 - sig))\\n    d_up = dy * act\\n    # GU is saved only for this backward. Overwriting both halves avoids\\n    # another N*2*INTER FP16 allocation and a separate concatenate operation.\\n    tl.store(gate_ptr, d_gate.to(tl.float16), mask=ok)\\n    tl.store(up_ptr, d_up.to(tl.float16), mask=ok)\\n\\n\\ndef _weight_shape(w):\\n    qs = getattr(w, \\\"quant_state\\\", None)\\n    if qs is None or getattr(qs, \\\"quant_type\\\", None) != \\\"nf4\\\":\\n        return None\\n    return tuple(getattr(qs, \\\"shape\\\", ()))\\n\\n\\ndef patch_grouped_dx_runtime_n():\\n    \\\"\\\"\\\"Install after patch_inputonly_mlp; fallback keeps original backward.\\\"\\\"\\\"\\n    import bitsandbytes.functional as bnbf\\n\\n    old = _InputOnly.backward\\n    if getattr(old, \\\"_soup_grouped_dx_runtime_n\\\", False):\\n        return True\\n\\n    @wraps(old)\\n    def backward(ctx, grad_out):\\n        (x,) = ctx.saved_tensors\\n        width = x.shape[-1]\\n        rows = x.numel() // width\\n        inter = _weight_shape(ctx.gate_w)\\n        up = _weight_shape(ctx.up_w)\\n        down = _weight_shape(ctx.down_w)\\n        valid = (isinstance(grad_out, torch.Tensor) and x.is_cuda\\n                 and grad_out.is_cuda and x.dtype == torch.float16\\n                 and grad_out.dtype in (torch.float16, torch.float32)\\n                 and x.is_contiguous() and grad_out.is_contiguous()\\n                 and x.ndim == 3 and grad_out.shape == x.shape\\n                 and 512 <= rows <= 2048 and width == 1536\\n                 and inter == up and inter is not None\\n                 and len(inter) == 2 and inter[1] == width\\n                 and inter[0] == 8960\\n                 and down == (width, inter[0])\\n                 and ctx.versions == (ctx.gate_w._version,\\n                     ctx.up_w._version, ctx.down_w._version))\\n        if not valid:\\n            COUNTS[\\\"fallback\\\"] += 1\\n            return old(ctx, grad_out)\\n\\n        intermediate = inter[0]\\n        with torch.autocast(\\\"cuda\\\", enabled=False):\\n            # 26 MiB, then release before the larger packed projection.\\n            wd = bnbf.dequantize_4bit(\\n                ctx.down_w.data, ctx.down_w.quant_state).to(torch.float16)\\n            dh = torch.mm(grad_out.view(rows, width).to(torch.float16), wd)\\n            del wd\\n\\n            # Two independent decoder weights share one contiguous allocation.\\n            wgu = torch.empty((2 * intermediate, width),\\n                              device=x.device, dtype=torch.float16)\\n            bnbf.dequantize_4bit(\\n                ctx.gate_w.data, ctx.gate_w.quant_state,\\n                out=wgu[:intermediate])\\n            bnbf.dequantize_4bit(\\n                ctx.up_w.data, ctx.up_w.quant_state,\\n                out=wgu[intermediate:])\\n            gu = torch.mm(x.view(rows, width), wgu.t())\\n            _soup_inplace_grouped_swiglu_vjp[\\n                (triton.cdiv(rows * intermediate, 1024),)\\n            ](gu, dh, rows * intermediate,\\n              INTER=intermediate, BLOCK=1024,\\n              num_warps=4, enable_fp_fusion=False)\\n            del dh\\n\\n            # One packed GEMM replaces separate dgate@Wgate and dup@Wup.\\n            # Its FP16 rounding is checked against reference in our oracle.\\n            dx = torch.mm(gu, wgu)\\n        COUNTS[\\\"grouped\\\"] += 1\\n        COUNTS[\\\"elements\\\"] += rows * intermediate\\n        return dx.view_as(x), None\\n\\n    backward._soup_grouped_dx_runtime_n = True\\n    _InputOnly.backward = staticmethod(backward)\\n    return True\\n\"},{\"name\":\"head_fused_t4.py\",\"sha\":\"1cc5926352899cf9f822a7d93fc9a2f0e6d255c8\",\"content\":\"\\\"\\\"\\\"Experimental Turing projection, sufficient statistics and scaled gradients.\\n\\nThis original kernel follows NVIDIA's documented m16n8k8 fragment mapping:\\nhttps://docs.nvidia.com/cuda/archive/11.7.1/parallel-thread-execution/index.html#warp-level-matrix-fragment-mma-1688\\n\\nA 128-thread CTA projects 64 tokens against 128 vocabulary rows. Cooperative\\nglobal loads stage a K=32 slice into 15 KiB of STATIC shared memory. Four warps\\nperform explicit FP16-input / FP32-accumulator MMA. Every logit is rounded to\\nFP16, matching an autocast linear, before stable max/sum/centered-sum/argmax\\nstatistics are formed. Only four scalars per token/vocabulary tile are written;\\nthere is no global [tokens, vocabulary] score allocation.\\n\\nThe recomputation entrypoint uses the very same projection instruction sequence\\nand writes an FP16 logit-gradient chunk directly. It rounds logits to FP16,\\nforms exp((score - saved maximum) - saved log normalizer) in FP32, and applies\\nthe actual upstream gradient divided by the loss denominator with a fused\\nmultiply-add before rounding to FP16. Keeping the two normalization terms\\nseparate avoids cancellation from a large common logit offset. For finite\\nprojected logits and valid centered state, ignored rows are positive zero,\\nincluding with a nonfinite upstream scale. A caller can form d(hidden) with\\nits usual FP16 matrix multiplication afterward.\\n\\nAll lane-sensitive memory and arithmetic live in one side-effecting scalar\\ninline-PTX block. The 40-half shared row stride makes each MMA fragment load\\nconflict-free in the standard 32-bank, four-byte shared-memory model. Staging\\nstores are not claimed conflict-free. Both stage barriers are uniform, and the\\nshared region is reused for statistics only after the last consumption barrier.\\n\\nTriton metadata.shared reports dynamic allocation (zero here). The cubin's\\n.nv.shared.soup_head_t4_fused_statistics section must report 15,360 bytes. This\\nstatic allocation is encoded in the cubin and needs no launch-time allocation.\\n\\nThese entrypoints require caller-owned autograd. Offline assembly and CPU\\nmodels cannot validate device execution, races, numerical drift or speed. Keep\\nthis experiment disabled until actual T4 correctness, compute-sanitizer and\\ntiming checks pass. Inputs must be finite, aligned contiguous FP16, and D must\\nbe a positive multiple of 32; valid target IDs or -100 are required. The\\nprojected FP16 logits must also be finite. The caller must route nonfinite or\\nnumerically saturated rows through its native implementation; these low-level\\nkernels do not preserve native exceptional-value behavior on their own.\\n\\\"\\\"\\\"\\n\\nimport torch\\nimport triton\\nimport triton.language as tl\\n\\n\\nSTATIC_SHARED_BYTES = (64 + 128) * 40 * 2\\nTILE_TOKENS = 64\\nTILE_VOCAB = 128\\n\\n\\ndef _projection_assembly_lines():\\n    \\\"\\\"\\\"One literally shared MMA sequence for forward and recomputation.\\n\\n    Operand positions1..8 are X,W,targets,M,startM,startV,D,V in both kernels.\\n    The statistics and gradient epilogues use their remaining operands.\\n    \\\"\\\"\\\"\\n    p = [\\n        \\\"{\\\",\\n        f\\\".shared .align 16 .b8 stage[{STATIC_SHARED_BYTES}];\\\",\\n        \\\".reg .u32 tid, lane, warp, wm, wn, group, pair, sm, idx, row, col, k, sk;\\\",\\n        \\\".reg .u32 sr, sc, addr, localrow, globalrow, column, target, winner, candidate, tile;\\\",\\n        \\\".reg .b32 a<4>, b<8>;\\\",\\n        \\\".reg .u32 winner0, winner1;\\\",\\n        \\\".reg .b16 half;\\\",\\n        \\\".reg .u64 offset, ptr, target64;\\\",\\n        \\\".reg .f32 c<64>, mx, total, centered_total, centered, expval, tmp;\\\",\\n        \\\".reg .f32 mx0, mx1, sum0, sum1, cs0, cs1, delta0, delta1, factor0, factor1;\\\",\\n        \\\".reg .f32 row_maximum, log_normalizer, scale, derivative, target_term;\\\",\\n        \\\".reg .pred valid, rowvalid, match, stop, leader, firsthalf, nonempty0, nonempty1;\\\",\\n        \\\".reg .pred supervised;\\\",\\n        \\\"mov.u32 tid, %tid.x;\\\",\\n        \\\"mov.u32 lane, %laneid;\\\",\\n        \\\"shr.u32 warp, tid, 5;\\\",\\n        \\\"and.b32 wm, warp, 1;\\\",\\n        \\\"shr.u32 wn, warp, 1;\\\",\\n        \\\"shr.u32 group, lane, 2;\\\",\\n        \\\"and.b32 pair, lane, 3;\\\",\\n        \\\"shl.b32 pair, pair, 1;\\\",\\n        \\\"mov.u32 sm, stage;\\\",\\n        \\\"shr.u32 tile, $6, 7;\\\",\\n        \\\"setp.eq.u32 leader, pair, 0;\\\",\\n    ]\\n    p += [f\\\"mov.f32 c{i}, 0f00000000;\\\" for i in range(64)]\\n    p += [\\\"mov.u32 k, 0;\\\", \\\"L_stage:\\\", \\\"setp.ge.u32 stop, k, $7;\\\", \\\"@stop bra L_epilogue;\\\"]\\n    # Every source half2 is loaded by exactly one lane. A and B each have\\n    # D-major contiguous pairs; the shared layout pads each K32 row to K40.\\n    for source, count, rowstart, rowlimit, shared_offset in [\\n        (\\\"$1\\\", 8, \\\"$5\\\", \\\"$4\\\", 0),\\n        (\\\"$2\\\", 16, \\\"$6\\\", \\\"$8\\\", 64 * 80),\\n    ]:\\n        for iteration in range(count):\\n            p += [\\n                f\\\"add.u32 idx, tid, {iteration * 128};\\\",\\n                \\\"shr.u32 sr, idx, 4;\\\",\\n                \\\"and.b32 sc, idx, 15;\\\",\\n                f\\\"add.u32 row, sr, {rowstart};\\\",\\n                \\\"shl.b32 col, sc, 1;\\\",\\n                \\\"add.u32 col, col, k;\\\",\\n                f\\\"setp.lt.u32 valid, row, {rowlimit};\\\",\\n                \\\"mul.wide.u32 offset, row, $7;\\\",\\n                \\\"cvt.u64.u32 ptr, col;\\\",\\n                \\\"add.u64 offset, offset, ptr;\\\",\\n                \\\"shl.b64 offset, offset, 1;\\\",\\n                f\\\"add.u64 ptr, {source}, offset;\\\",\\n                \\\"mov.u32 candidate, 0;\\\",\\n                \\\"@valid ld.global.b32 candidate, [ptr];\\\",\\n                \\\"mul.lo.u32 addr, sr, 80;\\\",\\n                \\\"mad.lo.u32 addr, sc, 4, addr;\\\",\\n                f\\\"add.u32 addr, addr, {shared_offset};\\\",\\n                \\\"add.u32 addr, addr, sm;\\\",\\n                \\\"st.shared.b32 [addr], candidate;\\\",\\n            ]\\n    p += [\\\"bar.sync 0;\\\"]\\n    for inner in range(4):\\n        # Four packed A registers cover two m16 fragments. Eight B registers\\n        # cover eight n8 fragments. All 32 lanes enter every MMA instruction.\\n        p += [f\\\"add.u32 sk, pair, {inner * 8};\\\", \\\"shl.b32 sk, sk, 1;\\\"]\\n        for i in range(4):\\n            p += [\\n                \\\"mad.lo.u32 row, wm, 32, group;\\\",\\n                f\\\"add.u32 row, row, {i * 8};\\\",\\n                \\\"mad.lo.u32 addr, row, 80, sk;\\\",\\n                \\\"add.u32 addr, addr, sm;\\\",\\n                f\\\"ld.shared.b32 a{i}, [addr];\\\",\\n            ]\\n        for j in range(8):\\n            p += [\\n                \\\"mad.lo.u32 row, wn, 64, group;\\\",\\n                f\\\"add.u32 row, row, {j * 8};\\\",\\n                \\\"mad.lo.u32 addr, row, 80, sk;\\\",\\n                f\\\"add.u32 addr, addr, {64 * 80};\\\",\\n                \\\"add.u32 addr, addr, sm;\\\",\\n                f\\\"ld.shared.b32 b{j}, [addr];\\\",\\n            ]\\n        for mb in range(2):\\n            for j in range(8):\\n                registers = \\\", \\\".join(f\\\"c{(mb * 8 + j) * 4 + i}\\\" for i in range(4))\\n                p.append(\\n                    \\\"mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 \\\"\\n                    f\\\"{{{registers}}}, {{a{mb*2}, a{mb*2+1}}}, {{b{j}}}, {{{registers}}};\\\"\\n                )\\n    # No lane can overwrite a stage while another warp still consumes it.\\n    p += [\\\"bar.sync 0;\\\", \\\"add.u32 k, k, 32;\\\", \\\"bra L_stage;\\\", \\\"L_epilogue:\\\"]\\n    return p\\n\\n\\ndef _statistics_epilogue_lines():\\n    p = []\\n    for q in range(4):\\n        mb, r = divmod(q, 2)\\n        p += [\\n            \\\"mad.lo.u32 localrow, wm, 32, group;\\\",\\n            f\\\"add.u32 localrow, localrow, {q * 8};\\\",\\n            \\\"add.u32 globalrow, localrow, $5;\\\",\\n            \\\"setp.lt.u32 rowvalid, globalrow, $4;\\\",\\n            \\\"mul.wide.u32 offset, globalrow, 8;\\\",\\n            \\\"add.u64 ptr, $3, offset;\\\",\\n            \\\"mov.u64 target64, 0xffffffffffffffff;\\\",\\n            \\\"@rowvalid ld.global.u64 target64, [ptr];\\\",\\n            \\\"cvt.u32.u64 target, target64;\\\",\\n            \\\"mov.f32 mx, 0fFF800000;\\\",\\n        ]\\n        entries = [(f\\\"c{(mb * 8 + j) * 4 + r * 2 + h}\\\", j * 8 + h) for j in range(8) for h in range(2)]\\n        for c, v in entries:\\n            p += [\\n                \\\"mad.lo.u32 column, wn, 64, pair;\\\",\\n                \\\"add.u32 column, column, $6;\\\",\\n                f\\\"add.u32 column, column, {v};\\\",\\n                \\\"setp.lt.u32 valid, column, $8;\\\",\\n                f\\\"cvt.rn.f16.f32 half, {c};\\\",\\n                f\\\"cvt.f32.f16 {c}, half;\\\",\\n                f\\\"selp.f32 {c}, {c}, 0fFF800000, valid;\\\",\\n                f\\\"max.f32 mx, mx, {c};\\\",\\n                \\\"setp.eq.u32 match, column, target;\\\",\\n                \\\"and.pred match, match, rowvalid;\\\",\\n                \\\"and.pred match, match, valid;\\\",\\n                \\\"mul.wide.u32 offset, globalrow, 4;\\\",\\n                \\\"add.u64 ptr, $13, offset;\\\",\\n                f\\\"@match st.global.f32 [ptr], {c};\\\",\\n            ]\\n        for delta in (1, 2):\\n            p += [f\\\"shfl.sync.bfly.b32 tmp, mx, {delta}, 31, 0xffffffff;\\\", \\\"max.f32 mx, mx, tmp;\\\"]\\n        p += [\\\"mov.f32 total, 0f00000000;\\\", \\\"mov.f32 centered_total, 0f00000000;\\\", \\\"mov.u32 winner, 2147483647;\\\"]\\n        for c, v in entries:\\n            p += [\\n                \\\"mad.lo.u32 column, wn, 64, pair;\\\",\\n                \\\"add.u32 column, column, $6;\\\",\\n                f\\\"add.u32 column, column, {v};\\\",\\n                \\\"setp.lt.u32 valid, column, $8;\\\",\\n                f\\\"sub.f32 centered, {c}, mx;\\\",\\n                \\\"selp.f32 centered, centered, 0f00000000, valid;\\\",\\n                \\\"mul.f32 tmp, centered, 0f3FB8AA3B;\\\",  # log2(e)\\n                \\\"ex2.approx.f32 expval, tmp;\\\",\\n                \\\"selp.f32 expval, expval, 0f00000000, valid;\\\",\\n                \\\"add.f32 total, total, expval;\\\",\\n                \\\"fma.rn.f32 centered_total, expval, centered, centered_total;\\\",\\n                f\\\"setp.eq.f32 match, {c}, mx;\\\",\\n                \\\"and.pred match, match, valid;\\\",\\n                \\\"selp.u32 candidate, column, 2147483647, match;\\\",\\n                \\\"min.u32 winner, winner, candidate;\\\",\\n            ]\\n        for delta in (1, 2):\\n            p += [\\n                f\\\"shfl.sync.bfly.b32 tmp, total, {delta}, 31, 0xffffffff;\\\",\\n                \\\"add.f32 total, total, tmp;\\\",\\n                f\\\"shfl.sync.bfly.b32 tmp, centered_total, {delta}, 31, 0xffffffff;\\\",\\n                \\\"add.f32 centered_total, centered_total, tmp;\\\",\\n                f\\\"shfl.sync.bfly.b32 candidate, winner, {delta}, 31, 0xffffffff;\\\",\\n                \\\"min.u32 winner, winner, candidate;\\\",\\n            ]\\n        p += [\\n            \\\"mad.lo.u32 addr, localrow, 2, wn;\\\",\\n            \\\"shl.b32 addr, addr, 4;\\\",\\n            \\\"add.u32 addr, addr, sm;\\\",\\n            \\\"@leader st.shared.f32 [addr], mx;\\\",\\n            \\\"@leader st.shared.f32 [addr+4], total;\\\",\\n            \\\"@leader st.shared.f32 [addr+8], centered_total;\\\",\\n            \\\"@leader st.shared.u32 [addr+12], winner;\\\",\\n        ]\\n    p += [\\n        \\\"bar.sync 0;\\\",\\n        \\\"setp.eq.u32 firsthalf, wn, 0;\\\",\\n        \\\"and.pred leader, leader, firsthalf;\\\",\\n        \\\"@!leader bra L_done;\\\",\\n    ]\\n    for q in range(4):\\n        p += [\\n            \\\"mad.lo.u32 localrow, wm, 32, group;\\\",\\n            f\\\"add.u32 localrow, localrow, {q * 8};\\\",\\n            \\\"add.u32 globalrow, localrow, $5;\\\",\\n            \\\"setp.lt.u32 rowvalid, globalrow, $4;\\\",\\n            \\\"shl.b32 addr, localrow, 5;\\\",\\n            \\\"add.u32 addr, addr, sm;\\\",\\n            \\\"ld.shared.f32 mx0, [addr];\\\",\\n            \\\"ld.shared.f32 sum0, [addr+4];\\\",\\n            \\\"ld.shared.f32 cs0, [addr+8];\\\",\\n            \\\"ld.shared.u32 winner0, [addr+12];\\\",\\n            \\\"ld.shared.f32 mx1, [addr+16];\\\",\\n            \\\"ld.shared.f32 sum1, [addr+20];\\\",\\n            \\\"ld.shared.f32 cs1, [addr+24];\\\",\\n            \\\"ld.shared.u32 winner1, [addr+28];\\\",\\n            \\\"max.f32 mx, mx0, mx1;\\\",\\n            \\\"setp.gt.f32 nonempty0, sum0, 0f00000000;\\\",\\n            \\\"setp.gt.f32 nonempty1, sum1, 0f00000000;\\\",\\n            \\\"sub.f32 delta0, mx0, mx;\\\",\\n            \\\"sub.f32 delta1, mx1, mx;\\\",\\n            \\\"selp.f32 delta0, delta0, 0f00000000, nonempty0;\\\",\\n            \\\"selp.f32 delta1, delta1, 0f00000000, nonempty1;\\\",\\n            \\\"mul.f32 factor0, delta0, 0f3FB8AA3B;\\\",\\n            \\\"mul.f32 factor1, delta1, 0f3FB8AA3B;\\\",\\n            \\\"ex2.approx.f32 factor0, factor0;\\\",\\n            \\\"ex2.approx.f32 factor1, factor1;\\\",\\n            \\\"mul.f32 total, factor0, sum0;\\\",\\n            \\\"fma.rn.f32 total, factor1, sum1, total;\\\",\\n            \\\"fma.rn.f32 cs0, delta0, sum0, cs0;\\\",\\n            \\\"fma.rn.f32 cs1, delta1, sum1, cs1;\\\",\\n            \\\"mul.f32 centered_total, factor0, cs0;\\\",\\n            \\\"fma.rn.f32 centered_total, factor1, cs1, centered_total;\\\",\\n            \\\"setp.eq.f32 match, mx0, mx;\\\",\\n            \\\"selp.u32 winner0, winner0, 2147483647, match;\\\",\\n            \\\"setp.eq.f32 match, mx1, mx;\\\",\\n            \\\"selp.u32 winner1, winner1, 2147483647, match;\\\",\\n            \\\"min.u32 winner, winner0, winner1;\\\",\\n            \\\"mad.lo.u32 idx, globalrow, $14, tile;\\\",\\n            \\\"mul.wide.u32 offset, idx, 4;\\\",\\n        ]\\n        for pointer, value in [(\\\"$9\\\", \\\"mx\\\"), (\\\"$10\\\", \\\"total\\\"), (\\\"$11\\\", \\\"centered_total\\\"), (\\\"$12\\\", \\\"winner\\\")]:\\n            p += [f\\\"add.u64 ptr, {pointer}, offset;\\\", f\\\"@rowvalid st.global.b32 [ptr], {value};\\\"]\\n    p += [\\\"L_done:\\\", \\\"mov.u32 $0, 0;\\\", \\\"}\\\"]\\n    return p\\n\\n\\ndef _gradient_epilogue_lines():\\n    \\\"\\\"\\\"Consume the common projection registers without materializing logits.\\n\\n    Additional operands are saved FP32 maximum and log normalizer, FP32 scale,\\n    and FP16 output. No collective instruction occurs here, so masked rows\\n    cannot affect barriers. Projected scores and centered state must be finite.\\n    \\\"\\\"\\\"\\n    p = [\\\"ld.global.f32 scale, [$11];\\\"]\\n    for q in range(4):\\n        mb, r = divmod(q, 2)\\n        p += [\\n            \\\"mad.lo.u32 localrow, wm, 32, group;\\\",\\n            f\\\"add.u32 localrow, localrow, {q * 8};\\\",\\n            \\\"add.u32 globalrow, localrow, $5;\\\",\\n            \\\"setp.lt.u32 rowvalid, globalrow, $4;\\\",\\n            \\\"mul.wide.u32 offset, globalrow, 8;\\\",\\n            \\\"add.u64 ptr, $3, offset;\\\",\\n            \\\"mov.u64 target64, 0xffffffffffffff9c;\\\",  # signed -100\\n            \\\"@rowvalid ld.global.u64 target64, [ptr];\\\",\\n            \\\"cvt.u32.u64 target, target64;\\\",\\n            \\\"setp.ne.u32 supervised, target, 0xffffff9c;\\\",\\n            \\\"mul.wide.u32 offset, globalrow, 4;\\\",\\n            \\\"add.u64 ptr, $9, offset;\\\",\\n            \\\"mov.f32 row_maximum, 0f00000000;\\\",\\n            \\\"@rowvalid ld.global.f32 row_maximum, [ptr];\\\",\\n            \\\"add.u64 ptr, $10, offset;\\\",\\n            \\\"mov.f32 log_normalizer, 0f00000000;\\\",\\n            \\\"@rowvalid ld.global.f32 log_normalizer, [ptr];\\\",\\n        ]\\n        for j in range(8):\\n            for h in range(2):\\n                c = f\\\"c{(mb * 8 + j) * 4 + r * 2 + h}\\\"\\n                p += [\\n                    \\\"mad.lo.u32 column, wn, 64, pair;\\\",\\n                    \\\"add.u32 column, column, $6;\\\",\\n                    f\\\"add.u32 column, column, {j * 8 + h};\\\",\\n                    \\\"setp.lt.u32 valid, column, $8;\\\",\\n                    \\\"and.pred valid, valid, rowvalid;\\\",\\n                    f\\\"cvt.rn.f16.f32 half, {c};\\\",\\n                    \\\"cvt.f32.f16 tmp, half;\\\",\\n                    \\\"sub.rn.f32 centered, tmp, row_maximum;\\\",\\n                    \\\"sub.rn.f32 centered, centered, log_normalizer;\\\",\\n                    \\\"mul.rn.f32 tmp, centered, 0f3FB8AA3B;\\\",\\n                    \\\"ex2.approx.f32 expval, tmp;\\\",\\n                    \\\"setp.eq.u32 match, column, target;\\\",\\n                    \\\"neg.f32 target_term, scale;\\\",\\n                    \\\"selp.f32 target_term, target_term, 0f00000000, match;\\\",\\n                    # Native log-softmax backward applies the scaled NLL\\n                    # gradient before its final multiply-add, including Inf.\\n                    \\\"fma.rn.f32 derivative, expval, scale, target_term;\\\",\\n                    # This guarantee requires finite scores/centered state.\\n                    # Exceptional ignored rows belong to the caller's native\\n                    # fallback because NaN probabilities times zero are NaN.\\n                    \\\"selp.f32 derivative, derivative, 0f00000000, supervised;\\\",\\n                    \\\"cvt.rn.f16.f32 half, derivative;\\\",\\n                    \\\"mul.wide.u32 offset, globalrow, $8;\\\",\\n                    \\\"cvt.u64.u32 ptr, column;\\\",\\n                    \\\"add.u64 offset, offset, ptr;\\\",\\n                    \\\"shl.b64 offset, offset, 1;\\\",\\n                    \\\"add.u64 ptr, $12, offset;\\\",\\n                    \\\"@valid st.global.b16 [ptr], half;\\\",\\n                ]\\n    return p + [\\\"mov.u32 $0, 0;\\\", \\\"}\\\"]\\n\\n\\ndef _fused_assembly(mode=\\\"statistics\\\"):\\n    if mode == \\\"statistics\\\":\\n        epilogue = _statistics_epilogue_lines()\\n    elif mode == \\\"gradient\\\":\\n        epilogue = _gradient_epilogue_lines()\\n    else:\\n        raise ValueError(f\\\"unknown fused projection mode: {mode}\\\")\\n    return \\\"\\\\n\\\".join(_projection_assembly_lines() + epilogue)\\n\\n\\n_FUSED_PTX = tl.constexpr(_fused_assembly())\\n_GRADIENT_PTX = tl.constexpr(_fused_assembly(\\\"gradient\\\"))\\n\\n\\n@triton.jit\\ndef soup_head_t4_fused_statistics(\\n    X, W, targets, partial_max, partial_sum, partial_centered_sum,\\n    partial_argmax, target_score, TOKENS,\\n    DEPTH: tl.constexpr, VOCAB: tl.constexpr, TILES: tl.constexpr,\\n):\\n    start_m = tl.program_id(0) * 64\\n    start_v = tl.program_id(1) * 128\\n    tl.inline_asm_elementwise(\\n        _FUSED_PTX,\\n        constraints=\\\"=r,l,l,l,r,r,r,r,r,l,l,l,l,l,r,~{memory}\\\",\\n        args=[X.to(tl.uint64), W.to(tl.uint64), targets.to(tl.uint64),\\n              TOKENS, start_m, start_v, DEPTH, VOCAB,\\n              partial_max.to(tl.uint64), partial_sum.to(tl.uint64),\\n              partial_centered_sum.to(tl.uint64), partial_argmax.to(tl.uint64),\\n              target_score.to(tl.uint64), TILES],\\n        dtype=tl.int32, is_pure=False, pack=1,\\n    )\\n\\n\\n@triton.jit\\ndef soup_head_t4_fused_logit_gradient(\\n    X, W, targets, row_max, log_normalizer, loss_gradient_over_denominator, output, TOKENS,\\n    DEPTH: tl.constexpr, VOCAB: tl.constexpr,\\n):\\n    start_m = tl.program_id(0) * 64\\n    start_v = tl.program_id(1) * 128\\n    tl.inline_asm_elementwise(\\n        _GRADIENT_PTX,\\n        constraints=\\\"=r,l,l,l,r,r,r,r,r,l,l,l,l,~{memory}\\\",\\n        args=[X.to(tl.uint64), W.to(tl.uint64), targets.to(tl.uint64),\\n              TOKENS, start_m, start_v, DEPTH, VOCAB,\\n              row_max.to(tl.uint64), log_normalizer.to(tl.uint64),\\n              loss_gradient_over_denominator.to(tl.uint64), output.to(tl.uint64)],\\n        dtype=tl.int32, is_pure=False, pack=1,\\n    )\\n\\n\\n@triton.jit\\ndef soup_head_t4_merge_statistics(\\n    targets, target_score, partial_max, partial_sum, partial_centered_sum,\\n    partial_argmax, token_loss, entropy, predictions, logsumexp, row_max, log_normalizer,\\n    TILES: tl.constexpr, BLOCK: tl.constexpr,\\n):\\n    row = tl.program_id(0)\\n    tile = tl.arange(0, BLOCK)\\n    active = tile < TILES\\n    offset = row * TILES + tile\\n    local_max = tl.load(partial_max + offset, active, other=-float(\\\"inf\\\"))\\n    maximum = tl.max(local_max, 0)\\n    delta = tl.where(active, local_max - maximum, 0.0)\\n    rescale = tl.exp(delta)\\n    local_sum = tl.load(partial_sum + offset, active, other=0.0)\\n    local_centered_sum = tl.load(partial_centered_sum + offset, active, other=0.0)\\n    total = tl.sum(rescale * local_sum, 0)\\n    centered_total = tl.sum(rescale * (local_centered_sum + delta * local_sum), 0)\\n    log_total = tl.log(total)\\n    lse = maximum + log_total\\n    local_winner = tl.load(partial_argmax + offset, active, other=2147483647)\\n    winner = tl.min(tl.where(active & (local_max == maximum), local_winner, 2147483647), 0)\\n    supervised = tl.load(targets + row) != -100\\n    correct_logit = tl.load(target_score + row)\\n    tl.store(token_loss + row, tl.where(supervised, -((correct_logit - maximum) - log_total), 0.0))\\n    tl.store(entropy + row, log_total - centered_total / total)\\n    tl.store(predictions + row, winner)\\n    tl.store(logsumexp + row, lse)\\n    tl.store(row_max + row, maximum)\\n    tl.store(log_normalizer + row, log_total)\\n\\n\\ndef _validate_projection_inputs(x, weight, targets):\\n    \\\"\\\"\\\"Shared metadata guards; inspecting them never synchronizes the device.\\\"\\\"\\\"\\n    if not all(isinstance(t, torch.Tensor) for t in (x, weight, targets)):\\n        raise TypeError(\\\"x, weight and targets must be tensors\\\")\\n    if x.ndim != 2 or weight.ndim != 2 or x.shape[1] != weight.shape[1]:\\n        raise ValueError(\\\"expected [M,D] and [V,D] with the same D\\\")\\n    rows, depth = x.shape\\n    vocab = weight.shape[0]\\n    if depth <= 0 or depth % 32 or vocab <= 0:\\n        raise ValueError(\\\"D must be a positive multiple of 32 and V must be positive\\\")\\n    if x.dtype != torch.float16 or weight.dtype != torch.float16 or targets.dtype != torch.int64:\\n        raise TypeError(\\\"x/weight must be FP16 and targets int64\\\")\\n    if targets.shape != (rows,):\\n        raise ValueError(\\\"targets must contain one already shifted label per row\\\")\\n    if x.device != weight.device or x.device != targets.device or not all(t.is_contiguous() for t in (x, weight, targets)):\\n        raise ValueError(\\\"all tensors must be contiguous and on the same device\\\")\\n    if x.data_ptr() % 4 or weight.data_ptr() % 4:\\n        raise ValueError(\\\"FP16 base pointers must be four-byte aligned\\\")\\n    if weight.requires_grad or (torch.is_grad_enabled() and x.requires_grad):\\n        raise ValueError(\\\"caller-owned autograd and frozen weights are required\\\")\\n    return rows, depth, vocab\\n\\n\\ndef fused_head_statistics_fp16(x, weight, targets):\\n    \\\"\\\"\\\"Return CE, entropy, prediction, LSE, maximum and log normalizer.\\n\\n    Research-only forward routine. The caller owns autograd/recomputation.\\n    Targets are int64 IDs in [0,V) or -100. Ignored CE is zero, while entropy\\n    and predictions still cover every input row. The FP32 maximum and centered\\n    log normalizer must be saved separately for recomputation; LSE is only a\\n    diagnostic output. Finite inputs and finite projected FP16 logits are\\n    required. The caller must detect exceptional or numerically saturated rows\\n    and use native arithmetic for them. No device-to-host validation or\\n    alternate training behavior is hidden inside this routine.\\n    \\\"\\\"\\\"\\n    rows, depth, vocab = _validate_projection_inputs(x, weight, targets)\\n    if not x.is_cuda:\\n        raise ValueError(\\\"the fused T4 assembly requires actual CUDA execution\\\")\\n    outputs = [torch.empty(rows, device=x.device, dtype=dtype) for dtype in\\n               (torch.float32, torch.float32, torch.int64, torch.float32, torch.float32, torch.float32)]\\n    if rows == 0:\\n        return tuple(outputs)\\n    tiles = triton.cdiv(vocab, TILE_VOCAB)\\n    partials = [torch.empty((rows, tiles), device=x.device, dtype=dtype) for dtype in\\n                (torch.float32, torch.float32, torch.float32, torch.int32)]\\n    # Exactly one matching logit writes each supervised target. Ignored rows\\n    # have no writer and use the initial zero; no atomic operation is needed.\\n    target_score = torch.zeros(rows, device=x.device, dtype=torch.float32)\\n    soup_head_t4_fused_statistics[(triton.cdiv(rows, TILE_TOKENS), tiles)](\\n        x, weight, targets, *partials, target_score, rows,\\n        DEPTH=depth, VOCAB=vocab, TILES=tiles, num_warps=4, num_stages=1,\\n    )\\n    soup_head_t4_merge_statistics[(rows,)](\\n        targets, target_score, *partials, *outputs,\\n        TILES=tiles, BLOCK=triton.next_power_of_2(tiles), num_warps=4,\\n    )\\n    return tuple(outputs)\\n\\n\\ndef fused_head_logit_gradient_fp16(\\n    x, weight, targets, row_max, log_normalizer, loss_gradient_over_denominator, out=None,\\n):\\n    \\\"\\\"\\\"Recompute logits and return their already-scaled FP16 gradient.\\n\\n    `row_max` and `log_normalizer` are the corresponding saved FP32 forward\\n    results, indexed in the same order as x/targets. They must remain separate\\n    to preserve centered normalization. `loss_gradient_over_denominator` is one\\n    FP32 value containing the real upstream autograd gradient divided by the\\n    loss's denominator, including any AMP scale. Pass that value before any\\n    FP16 conversion. Projected scores and the saved centered state must be\\n    finite. Under that precondition, ignored rows are exactly positive zero,\\n    including when the scale is Inf or NaN for an all-ignored loss. Exceptional\\n    scores and numerically saturated rows require the caller's native fallback.\\n\\n    The optional output is contiguous FP16 [M,V]; it may be two-byte aligned\\n    because stores are individual halfwords. It must not overlap any input.\\n    Caller owns autograd. This function does not calculate d(hidden).\\n    \\\"\\\"\\\"\\n    rows, depth, vocab = _validate_projection_inputs(x, weight, targets)\\n    for name, tensor, count in [\\n        (\\\"row_max\\\", row_max, rows),\\n        (\\\"log_normalizer\\\", log_normalizer, rows),\\n        (\\\"loss_gradient_over_denominator\\\", loss_gradient_over_denominator, 1),\\n    ]:\\n        if not isinstance(tensor, torch.Tensor) or tensor.dtype != torch.float32:\\n            raise TypeError(f\\\"{name} must be an FP32 tensor\\\")\\n        if tensor.device != x.device or not tensor.is_contiguous() or tensor.numel() != count:\\n            raise ValueError(f\\\"{name} must be contiguous, correctly sized and on the input device\\\")\\n        if torch.is_grad_enabled() and tensor.requires_grad:\\n            raise ValueError(\\\"caller-owned autograd is required for saved centered state and scale\\\")\\n    if row_max.shape != (rows,) or log_normalizer.shape != (rows,):\\n        raise ValueError(\\\"row_max and log_normalizer must have shape [M]\\\")\\n    if out is not None:\\n        if (not isinstance(out, torch.Tensor) or out.dtype != torch.float16\\n                or out.device != x.device or out.shape != (rows, vocab)\\n                or not out.is_contiguous() or out.requires_grad):\\n            raise ValueError(\\\"out must be contiguous FP16 [M,V] on the input device without autograd\\\")\\n        # All supported views are contiguous, so exact byte intervals suffice.\\n        # Prevent cross-CTA races if a caller aliases scores with x or weight.\\n        if out.numel():\\n            begin, end = out.data_ptr(), out.data_ptr() + out.numel() * out.element_size()\\n            for tensor in (x, weight, targets, row_max, log_normalizer, loss_gradient_over_denominator):\\n                other_begin = tensor.data_ptr()\\n                other_end = other_begin + tensor.numel() * tensor.element_size()\\n                if tensor.numel() and begin < other_end and other_begin < end:\\n                    raise ValueError(\\\"out must not overlap any input tensor\\\")\\n    if not x.is_cuda:\\n        raise ValueError(\\\"the fused T4 assembly requires actual CUDA execution\\\")\\n    if out is None:\\n        out = torch.empty((rows, vocab), device=x.device, dtype=torch.float16)\\n    if rows:\\n        soup_head_t4_fused_logit_gradient[(triton.cdiv(rows, TILE_TOKENS), triton.cdiv(vocab, TILE_VOCAB))](\\n            x, weight, targets, row_max, log_normalizer, loss_gradient_over_denominator, out, rows,\\n            DEPTH=depth, VOCAB=vocab, num_warps=4, num_stages=1,\\n        )\\n    return out\\n\"}]")
for item in items:
 name=item['name'];assert name.endswith('.py') and '/' not in name and name not in ('swiglu.py','test_kernels.py','conftest.py')
 raw=item['content'].encode('utf-8')
 digest=hashlib.sha1(('blob '+str(len(raw))).encode()+bytes([0])+raw).hexdigest()
 assert digest==item['sha'],(name,digest,item['sha'])
 ast.parse(item['content'],filename=name)
 (root/name).write_bytes(raw)
print('Source chunk 1/4 verified: '+str(len(items))+' files')


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.
